In [1]:
import sqlite3
import pandas as pd

# Tạo database trong bộ nhớ
conn = sqlite3.connect(':memory:')

# Tạo bảng student
conn.execute("""
CREATE TABLE student(
    student_id INT,
    name TEXT,
    class TEXT,
    course_id INT,
    score REAL
)
""")

# Chèn dữ liệu
conn.executemany("""
INSERT INTO student VALUES (?,?,?,?,?)
""",[
(1,"Nguyen Minh Hoang","May Tinh",12,6.7),
(2,"Tran Thi Lan","Kinh Te",34,9.2),
(3,"Pham Van Nam","Toan Tin",None,7.9),
(4,"Le Thanh Huyen","Toan Tin",20,7.2),
(5,"Vu Quoc Anh","May Tinh",24,8.0),
(6,"Dang Thuy Linh","May Tinh",24,5.5),
(7,"Bui Tien Dung","Kinh Te",34,9.2),
(8,"Ho Ngoc Mai","Toan Tin",20,8.8),
(9,"Duong Huu Phuc","Kinh Te",None,7.2),
(10,"Cao Thi Hanh","May Tinh",None,7.0)
])

# Tạo bảng course
conn.execute("""
CREATE TABLE course(
    id INT,
    course_name TEXT
)
""")

# Chèn dữ liệu
conn.executemany("""
INSERT INTO course VALUES (?,?)
""",[
(12,"Giai tich"),
(34,"Thong ke"),
(26,"Tin hoc")
])

##1. Kết nối hai bảng

a, Sử dụng tích Decartes.

In [2]:
query = """
SELECT *
FROM student
CROSS JOIN course
"""

pd.read_sql(query,conn)

,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,12,Giai tich
1,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,34,Thong ke
2,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,26,Tin hoc
3,2,Tran Thi Lan,Kinh Te,34.0,9.2,12,Giai tich
4,2,Tran Thi Lan,Kinh Te,34.0,9.2,34,Thong ke
5,2,Tran Thi Lan,Kinh Te,34.0,9.2,26,Tin hoc
6,3,Pham Van Nam,Toan Tin,NaN,7.9,12,Giai tich
7,3,Pham Van Nam,Toan Tin,NaN,7.9,34,Thong ke
8,3,Pham Van Nam,Toan Tin,NaN,7.9,26,Tin hoc
9,4,Le Thanh Huyen,Toan Tin,20.0,7.2,12,Giai tich


KL: Với tích Decartes mỗi dòng của bảng student được kết hợp với tất cả các dòng của bảng course. Vì bảng student có 10 sinh viên và bảng course có 3 môn học, nên bảng kết quả có 30 dòng. Điều này có nghĩa là mỗi sinh viên xuất hiện 3 lần, tương ứng với 3 môn học khác nhau.

b, Sử dụng JOIN:

INNER JOIN

In [3]:
query = """
SELECT *
FROM student
INNER JOIN course
ON student.course_id = course.id
"""

pd.read_sql(query,conn)

,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,12,Giai tich
1,2,Tran Thi Lan,Kinh Te,34,9.2,34,Thong ke
2,7,Bui Tien Dung,Kinh Te,34,9.2,34,Thong ke


KL: INNER JOIN chỉ giữ lại các dòng có khóa liên kết khớp ở cả hai bảng, do đó các sinh viên không có course_id hoặc không khớp với bảng course sẽ bị loại khỏi kết quả.
+ Sinh viên có course_id = NULL nên không được xuất hiện trong bảng
+ Những sinh viên đăng ký các môn Giải tích (12) và Thống kê (34) được ghép thành công.
+ Môn Tin học (26) không có sinh viên nào đăng ký nên không xuất hiện ở bảng

LEFT JOIN

In [4]:
query = """
SELECT *
FROM student
LEFT JOIN course
ON student.course_id = course.id
"""

pd.read_sql(query,conn)

,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,12.0,Giai tich
1,2,Tran Thi Lan,Kinh Te,34.0,9.2,34.0,Thong ke
2,3,Pham Van Nam,Toan Tin,NaN,7.9,NaN,None
3,4,Le Thanh Huyen,Toan Tin,20.0,7.2,NaN,None
4,5,Vu Quoc Anh,May Tinh,24.0,8.0,NaN,None
5,6,Dang Thuy Linh,May Tinh,24.0,5.5,NaN,None
6,7,Bui Tien Dung,Kinh Te,34.0,9.2,34.0,Thong ke
7,8,Ho Ngoc Mai,Toan Tin,20.0,8.8,NaN,None
8,9,Duong Huu Phuc,Kinh Te,NaN,7.2,NaN,None
9,10,Cao Thi Hanh,May Tinh,NaN,7.0,NaN,None


KL: Truy vấn sử dụng LEFT JOIN giữ lại toàn bộ dữ liệu của bảng student và ghép thêm thông tin từ bảng course khi điều kiện student.course_id = course.id thỏa mãn.

Kết quả truy vấn cho thấy toàn bộ 10 sinh viên của bảng student đều xuất hiện trong bảng kết quả. Những sinh viên có course_id trùng với id trong bảng course sẽ được ghép thêm thông tin môn học.
+ Nguyen Minh Hoang (course_id = 12) → ghép với môn Giải tích.
+ Tran Thi Lan và Bui Tien Dung (course_id = 34) → ghép với môn Thống kê.
+ Các sinh viên còn lại có course_id = NULL hoặc course_id không tồn tại trong bảng course (20, 24) nên các cột id và course_name nhận giá trị NULL/None.

 RIGHT JOIN

In [5]:
query = """
SELECT *
FROM course
LEFT JOIN student
ON student.course_id = course.id
"""

pd.read_sql(query,conn)

,id,course_name,student_id,name,class,course_id,score
0,12,Giai tich,1.0,Nguyen Minh Hoang,May Tinh,12.0,6.7
1,34,Thong ke,2.0,Tran Thi Lan,Kinh Te,34.0,9.2
2,34,Thong ke,7.0,Bui Tien Dung,Kinh Te,34.0,9.2
3,26,Tin hoc,NaN,None,None,NaN,NaN


KL: Truy vấn RIGHT JOIN giữ toàn bộ dữ liệu của bảng bên phải (course) và ghép thêm thông tin từ bảng student khi điều kiện student.course_id = course.id thỏa mãn.
+ Môn Giải tích (id = 12) có 1 sinh viên đăng ký: Nguyen Minh Hoang.
+ Môn Thống kê (id = 34) có 2 sinh viên đăng ký: Tran Thi Lan và Bui Tien Dung nên xuất hiện 2 dòng.
+ Môn Tin học (id = 26) không có sinh viên nào đăng ký, nên các cột thông tin của student có giá trị NULL/None.

FULL OUTER JOIN.

In [6]:
query = """
SELECT s.student_id, s.name, s.class, s.score, c.id AS course_id, c.course_name
FROM student AS s
LEFT JOIN course AS c
ON s.course_id = c.id

UNION

SELECT s.student_id, s.name, s.class, s.score, c.id AS course_id, c.course_name
FROM course AS c
LEFT JOIN student AS s
ON s.course_id = c.id
"""
pd.read_sql(query,conn)

,student_id,name,class,score,course_id,course_name
0,NaN,None,None,NaN,26.0,Tin hoc
1,1.0,Nguyen Minh Hoang,May Tinh,6.7,12.0,Giai tich
2,2.0,Tran Thi Lan,Kinh Te,9.2,34.0,Thong ke
3,3.0,Pham Van Nam,Toan Tin,7.9,NaN,None
4,4.0,Le Thanh Huyen,Toan Tin,7.2,NaN,None
5,5.0,Vu Quoc Anh,May Tinh,8.0,NaN,None
6,6.0,Dang Thuy Linh,May Tinh,5.5,NaN,None
7,7.0,Bui Tien Dung,Kinh Te,9.2,34.0,Thong ke
8,8.0,Ho Ngoc Mai,Toan Tin,8.8,NaN,None
9,9.0,Duong Huu Phuc,Kinh Te,7.2,NaN,None


KL: Truy vấn FULL OUTER JOIN, tức là giữ lại tất cả bản ghi của cả hai bảng dù có khớp hay không.
+ Các sinh viên có course_id trùng với bảng course được ghép đúng với môn học:    
        + Nguyen Minh Hoang → Giải tích
        
        + Tran Thi Lan và Bui Tien Dung → Thống kê

+ Các sinh viên không có course_id hoặc không khớp với bảng course (course_id NULL, 20, 24) vẫn xuất hiện nhưng course_name là NULL.
+ Môn Tin học (id = 26) không có sinh viên đăng ký, nên vẫn xuất hiện trong kết quả nhưng các cột thông tin sinh viên là NULL.

##2. Hãy cập nhật những giá trị course_id còn thiếu trong bảng student bằng câu lệnh SQL,trong đó các giá trị được điền là những giá trị nằm trong bảng course và loại bỏ những bản ghi tham gia những môn học không tồn tại bảng course. Sau đó hãy cho biết:

 Cập nhật những giá trị course_id còn thiếu trong bảng student bằng câu lệnh SQL

In [7]:
conn.execute("UPDATE student SET course_id = 26 WHERE course_id IS NULL AND class = 'May Tinh'")
conn.execute("UPDATE student SET course_id = 12 WHERE course_id IS NULL AND class = 'Toan Tin'")
conn.execute("UPDATE student SET course_id = 34 WHERE course_id IS NULL AND class = 'Kinh Te'")

KL: Các câu lệnh này giúp loại bỏ giá trị NULL của course_id, bằng cách tự động gán môn học phù hợp theo từng lớp sinh viên, từ đó giúp việc JOIN giữa bảng student và course đầy đủ hơn.

Ở đây ghép:
+ course_id = 26 ghép với lớp máy tính
+ course_id = 26 ghép với lớp toán tin
+ course_id = 26 ghép với lớp kinh tế|

Loại bỏ những
bản ghi tham gia những môn học không tồn tại bảng course

In [8]:
conn.execute("""
DELETE FROM student
WHERE course_id NOT IN (SELECT id FROM course)
""")

KL: Câu lệnh này dùng để xóa các sinh viên có course_id không tồn tại trong bảng course.
+ Các course_id hợp lệ trong bảng course là 12, 34, 26.
+ Những sinh viên có course_id khác các giá trị trên (ví dụ 20, 24) sẽ bị xóa khỏi bảng.

a. Tổng số sinh viên, điểm trung bình của từng lớp


In [9]:
query = """
SELECT class,
       COUNT(student_id) AS total_students,
       ROUND(AVG(score),2) AS avg_score
FROM student
GROUP BY class
"""
pd.read_sql(query,conn)

,class,total_students,avg_score
0,Kinh Te,3,8.53
1,May Tinh,2,6.85
2,Toan Tin,1,7.90


KL: Truy vấn sử dụng GROUP BY class để nhóm sinh viên theo từng lớp, sau đó dùng COUNT(student_id) tính tổng số sinh viên của mỗi lớp và dùng AVG(score) để tính điểm trung bình của lớp
+ Kinh Te: có 3 sinh viên, điểm trung bình 8.53 → lớp có điểm trung bình cao nhất.
+ May Tinh: có 2 sinh viên, điểm trung bình 6.85.
+ Toan Tin: có 1 sinh viên, điểm trung bình 7.90.

b. Tổng số sinh viên, điểm trung bình của từng môn học

In [10]:
query = """
SELECT course.course_name,
       COUNT(student.student_id) AS total_students,
       ROUND(AVG(student.score),2) AS avg_score
FROM student
JOIN course
ON student.course_id = course.id
GROUP BY course.course_name
"""
pd.read_sql(query,conn)

,course_name,total_students,avg_score
0,Giai tich,2,7.30
1,Thong ke,3,8.53
2,Tin hoc,1,7.00


KL: Truy vấn sử dụng JOIN để kết nối bảng student và course theo student.course_id = course.id, sau đó dùng GROUP BY course.course_name để thống kê theo từng môn học
+ Giải tích: có 2 sinh viên, điểm trung bình 7.30
+ Thống kê: có 3 sinh viên, điểm trung bình 8.53 → vừa có nhiều sinh viên nhất vừa có điểm trung bình cao nhất
+ Tin học: có 1 sinh viên, điểm trung bình 7.00


c. Phân loại thi đua theo số điểm của từng môn học biết:
+ Điểm TB ≥ 9.0: Xuất sắc.
+ 6.0 ≤ Điểm TB ≤ 8.9: Tốt.
+ Điểm TB < 6.0: Kém.

In [11]:
query = """
SELECT course.course_name,
       ROUND(AVG(student.score),2) AS avg_score,
       CASE
            WHEN AVG(student.score) >= 9.0 THEN 'Xuat sac'
            WHEN AVG(student.score) BETWEEN 6.0 AND 8.9 THEN 'Tot'
            WHEN AVG(student.score) < 6.0 THEN 'Kem'
       END AS classification
FROM student
JOIN course
ON student.course_id = course.id
GROUP BY course.course_name
"""
pd.read_sql(query,conn)

,course_name,avg_score,classification
0,Giai tich,7.30,Tot
1,Thong ke,8.53,Tot
2,Tin hoc,7.00,Tot


KL: Truy vấn sử dụng JOIN để kết nối hai bảng student và course, sau đó tính điểm trung bình (AVG(score)) của từng môn học và dùng CASE để phân loại kết quả học tập với:
- Điểm TB ≥ 9.0: Xuất sắc.
- 6.0 ≤ Điểm TB ≤ 8.9: Tốt.
- Điểm TB < 6.0: Kém.


Kết quả:
+ Giải tích: điểm trung bình 7.30 → xếp loại Tốt.
+ Thống kê: điểm trung bình 8.53 → xếp loại Tốt (điểm trung bình cao nhất).
+ Tin học: điểm trung bình 7.00 → xếp loại Tốt.


## 3. Hãy xếp hạng sinh viên thông qua:

a. Điểm số.


In [12]:
query = """
SELECT student_id, name, class, course_id, score,
       RANK() OVER (ORDER BY score DESC) AS rank_score
FROM student
"""
pd.read_sql(query, conn)

,student_id,name,class,course_id,score,rank_score
0,2,Tran Thi Lan,Kinh Te,34,9.2,1
1,7,Bui Tien Dung,Kinh Te,34,9.2,1
2,3,Pham Van Nam,Toan Tin,12,7.9,3
3,9,Duong Huu Phuc,Kinh Te,34,7.2,4
4,10,Cao Thi Hanh,May Tinh,26,7.0,5
5,1,Nguyen Minh Hoang,May Tinh,12,6.7,6


KL: Sử dụng hàm cửa sổ RANK() để xếp hạng sinh viên theo điểm score giảm dần (ORDER BY score DESC). Sinh viên có điểm cao hơn sẽ có thứ hạng nhỏ hơn.
+ Tran Thi Lan và Bui Tien Dung có điểm 9.2 (cao nhất) nên cùng xếp hạng 1.
+ Pham Van Nam có điểm 7.9 nên đứng hạng 3 .
+ Duong Huu Phuc có điểm 7.2 nên xếp hạng 4.
+ Cao Thi Hanh có điểm 7.0 nên xếp hạng 5.
+ Nguyen Minh Hoang có điểm 6.7 nên xếp hạng 6.

 top 3 sinh viện đạt thứ hạng cao nhất

In [13]:
query = """
SELECT *
FROM (
    SELECT student_id, name, class, course_id, score,
           RANK() OVER (ORDER BY score DESC) AS rank_score
    FROM student
)
WHERE rank_score <= 3
"""
pd.read_sql(query, conn)

,student_id,name,class,course_id,score,rank_score
0,2,Tran Thi Lan,Kinh Te,34,9.2,1
1,7,Bui Tien Dung,Kinh Te,34,9.2,1
2,3,Pham Van Nam,Toan Tin,12,7.9,3


KL: Truy vấn sử dụng hàm cửa sổ RANK() để xếp hạng sinh viên theo điểm giảm dần, sau đó dùng truy vấn con để lọc các sinh viên có rank_score ≤ 3, tức là những sinh viên có điểm cao nhất trong top 3.
+ Tran Thi Lan và Bui Tien Dung có điểm 9.2, cao nhất nên cùng xếp hạng 1.
+ Pham Van Nam có điểm 7.9 nên xếp hạng 3.


 top 3 sinh viên đạt thứ hạng thấp nhất

In [14]:
query = """
SELECT *
FROM (
    SELECT student_id, name, class, course_id, score,
           RANK() OVER (ORDER BY score ASC) AS rank_score
    FROM student
)
WHERE rank_score <= 3
"""
pd.read_sql(query, conn)

,student_id,name,class,course_id,score,rank_score
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,1
1,10,Cao Thi Hanh,May Tinh,26,7.0,2
2,9,Duong Huu Phuc,Kinh Te,34,7.2,3


KL: Truy vấn sử dụng RANK() OVER (ORDER BY score ASC) để xếp hạng sinh viên theo điểm tăng dần, tức là sinh viên điểm thấp hơn sẽ có thứ hạng cao hơn. Sau đó, truy vấn ngoài chọn các bản ghi có rank_score ≤ 3, tức là 3 sinh viên có điểm thấp nhất.
+ Nguyen Minh Hoang có điểm 6.7 → hạng 1 (thấp nhất).
+ Cao Thi Hanh có điểm 7.0 → hạng 2.
+ Duong Huu Phuc có điểm 7.2 → hạng 3.

b. Điểm số theo lớp học.

In [15]:
query = """
SELECT student_id, name, class, score,
       RANK() OVER (PARTITION BY class ORDER BY score DESC) AS rank_class
FROM student
"""
pd.read_sql(query, conn)

,student_id,name,class,score,rank_class
0,2,Tran Thi Lan,Kinh Te,9.2,1
1,7,Bui Tien Dung,Kinh Te,9.2,1
2,9,Duong Huu Phuc,Kinh Te,7.2,3
3,10,Cao Thi Hanh,May Tinh,7.0,1
4,1,Nguyen Minh Hoang,May Tinh,6.7,2
5,3,Pham Van Nam,Toan Tin,7.9,1


KL: Truy vấn sử dụng RANK() kết hợp với PARTITION BY class để xếp hạng sinh viên trong từng lớp riêng biệt theo điểm giảm dần (ORDER BY score DESC). Mỗi lớp sẽ có một bảng xếp hạng riêng.

Lớp Kinh Tế:

+ Tran Thi Lan và Bui Tien Dung có điểm 9.2 → đồng hạng 1.
+ Duong Huu Phuc có điểm 7.2 → hạng 3.

Lớp Máy Tính:
+ Cao Thi Hanh 7.0 → hạng 1.
+ Nguyen Minh Hoang 6.7 → hạng 2.

Lớp Toán Tin:
+ Pham Van Nam 7.9 → hạng 1 (chỉ có 1 sinh viên).

 top 3 sinh viện đạt thứ hạng cao nhất mỗi lớp

In [16]:
query = """
SELECT *
FROM (
    SELECT student_id, name, class, score,
           RANK() OVER (PARTITION BY class ORDER BY score DESC) AS rank_class
    FROM student
)
WHERE rank_class <= 3
"""
pd.read_sql(query, conn)

,student_id,name,class,score,rank_class
0,2,Tran Thi Lan,Kinh Te,9.2,1
1,7,Bui Tien Dung,Kinh Te,9.2,1
2,9,Duong Huu Phuc,Kinh Te,7.2,3
3,10,Cao Thi Hanh,May Tinh,7.0,1
4,1,Nguyen Minh Hoang,May Tinh,6.7,2
5,3,Pham Van Nam,Toan Tin,7.9,1


KL: Truy vấn sử dụng RANK() với PARTITION BY class để xếp hạng sinh viên theo điểm giảm dần trong từng lớp, sau đó dùng truy vấn ngoài để lọc các sinh viên có rank_class ≤ 3, tức là top 3 sinh viên có điểm cao nhất trong mỗi lớp.

Lớp Kinh Tế:
+ Tran Thi Lan và Bui Tien Dung (9.2) → hạng 1.
+ Duong Huu Phuc (7.2) → hạng 3.

Lớp Máy Tính:
+ Cao Thi Hanh (7.0) → hạng 1.
+ Nguyen Minh Hoang (6.7) → hạng 2.

Lớp Toán Tin:
+ Pham Van Nam (7.9) → hạng 1.

 top 3 sinh viện đạt thứ hạng thấp nhất mỗi lớp

In [17]:
query = """
SELECT *
FROM (
    SELECT student_id, name, class, score,
           RANK() OVER (PARTITION BY class ORDER BY score ASC) AS rank_class
    FROM student
)
WHERE rank_class <= 3
"""
pd.read_sql(query, conn)

,student_id,name,class,score,rank_class
0,9,Duong Huu Phuc,Kinh Te,7.2,1
1,2,Tran Thi Lan,Kinh Te,9.2,2
2,7,Bui Tien Dung,Kinh Te,9.2,2
3,1,Nguyen Minh Hoang,May Tinh,6.7,1
4,10,Cao Thi Hanh,May Tinh,7.0,2
5,3,Pham Van Nam,Toan Tin,7.9,1


KL: Truy vấn sử dụng RANK() với PARTITION BY class để xếp hạng sinh viên theo điểm tăng dần (ORDER BY score ASC) trong từng lớp, sau đó lọc các sinh viên có rank_class ≤ 3, tức là top 3 những sinh viên có điểm thấp nhất trong mỗi lớp

Lớp Kinh Tế:
+ Duong Huu Phuc (7.2) → hạng 1 (điểm thấp nhất).
+ Tran Thi Lan (9.2) và Bui Tien Dung (9.2) → đồng hạng 2.

Lớp Máy Tính:
+ Nguyen Minh Hoang (6.7) → hạng 1.
+ Cao Thi Hanh (7.0) → hạng 2.

Lớp Toán Tin:
+ Pham Van Nam (7.9) → hạng 1 (chỉ có 1 sinh viên).

c. Điểm số theo mã môn học.

In [18]:
query = """
SELECT s.student_id,
       s.name,
       s.class,
       s.course_id,
       c.course_name,
       s.score,
       RANK() OVER (PARTITION BY s.course_id ORDER BY s.score DESC) AS rank_course
FROM student s
JOIN course c
ON s.course_id = c.id
ORDER BY s.course_id, rank_course
"""
pd.read_sql(query, conn)

,student_id,name,class,course_id,course_name,score,rank_course
0,3,Pham Van Nam,Toan Tin,12,Giai tich,7.9,1
1,1,Nguyen Minh Hoang,May Tinh,12,Giai tich,6.7,2
2,10,Cao Thi Hanh,May Tinh,26,Tin hoc,7.0,1
3,2,Tran Thi Lan,Kinh Te,34,Thong ke,9.2,1
4,7,Bui Tien Dung,Kinh Te,34,Thong ke,9.2,1
5,9,Duong Huu Phuc,Kinh Te,34,Thong ke,7.2,3


KL: Truy vấn sử dụng RANK() với PARTITION BY s.course_id để xếp hạng sinh viên theo điểm giảm dần trong từng môn học. Sau đó kết quả được sắp xếp theo course_id và thứ hạng (rank_course) để dễ quan sát.

Môn Giải tích (course_id = 12):
+ Pham Van Nam (7.9) → hạng 1.
+ Nguyen Minh Hoang (6.7) → hạng 2.

Môn Tin học (course_id = 26):
+ Cao Thi Hanh (7.0) → hạng 1.

Môn Thống kê (course_id = 34):
+ Tran Thi Lan và Bui Tien Dung (9.2) → đồng hạng 1.
+ Duong Huu Phuc (7.2) → hạng 3.

 top 3 sinh viện đạt thứ hạng cao nhất mỗi môn


In [19]:
query = """
SELECT *
FROM (
    SELECT s.student_id,
           s.name,
           s.class,
           s.course_id,
           c.course_name,
           s.score,
           RANK() OVER (PARTITION BY s.course_id ORDER BY s.score DESC) AS rank_course
    FROM student s
    JOIN course c
    ON s.course_id = c.id
)
WHERE rank_course <= 3
ORDER BY course_id, rank_course
"""
pd.read_sql(query, conn)

,student_id,name,class,course_id,course_name,score,rank_course
0,3,Pham Van Nam,Toan Tin,12,Giai tich,7.9,1
1,1,Nguyen Minh Hoang,May Tinh,12,Giai tich,6.7,2
2,10,Cao Thi Hanh,May Tinh,26,Tin hoc,7.0,1
3,2,Tran Thi Lan,Kinh Te,34,Thong ke,9.2,1
4,7,Bui Tien Dung,Kinh Te,34,Thong ke,9.2,1
5,9,Duong Huu Phuc,Kinh Te,34,Thong ke,7.2,3


KL: Truy vấn sử dụng RANK() với PARTITION BY s.course_id để xếp hạng sinh viên theo điểm giảm dần trong từng môn học, sau đó lọc các bản ghi có rank_course ≤ 3, tức là top 3 sinh viên có điểm cao nhất của mỗi môn.

Môn Giải tích (course_id = 12):
+ Pham Van Nam (7.9) → hạng 1
+ Nguyen Minh Hoang (6.7) → hạng 2

Môn Tin học (course_id = 26):
+ Cao Thi Hanh (7.0) → hạng 1

Môn Thống kê (course_id = 34):
+ Tran Thi Lan và Bui Tien Dung (9.2) → đồng hạng 1
+ Duong Huu Phuc (7.2) → hạng 3


 top 3 sinh viên đạt thứ hạng thấp nhất mỗi môn

In [20]:
query = """
SELECT *
FROM (
    SELECT s.student_id,
           s.name,
           s.class,
           s.course_id,
           c.course_name,
           s.score,
           RANK() OVER (PARTITION BY s.course_id ORDER BY s.score ASC) AS rank_course
    FROM student s
    JOIN course c
    ON s.course_id = c.id
)
WHERE rank_course <= 3
ORDER BY course_id, rank_course
"""
pd.read_sql(query, conn)

,student_id,name,class,course_id,course_name,score,rank_course
0,1,Nguyen Minh Hoang,May Tinh,12,Giai tich,6.7,1
1,3,Pham Van Nam,Toan Tin,12,Giai tich,7.9,2
2,10,Cao Thi Hanh,May Tinh,26,Tin hoc,7.0,1
3,9,Duong Huu Phuc,Kinh Te,34,Thong ke,7.2,1
4,2,Tran Thi Lan,Kinh Te,34,Thong ke,9.2,2
5,7,Bui Tien Dung,Kinh Te,34,Thong ke,9.2,2


KL: Truy vấn sử dụng RANK() với PARTITION BY s.course_id và ORDER BY s.score ASC để xếp hạng sinh viên theo điểm tăng dần trong từng môn học, sau đó lọc các sinh viên có rank_course ≤ 3, tức là top 3 sinh viên có điểm thấp nhất trong mỗi môn.

Môn Giải tích (course_id = 12):
+ Nguyen Minh Hoang (6.7) → hạng 1 (thấp nhất)
+ Pham Van Nam (7.9) → hạng 2

Môn Tin học (course_id = 26):
+ Cao Thi Hanh (7.0) → hạng 1 (chỉ có 1 sinh viên)

Môn Thống kê (course_id = 34):
+ Duong Huu Phuc (7.2) → hạng 1 (thấp nhất)
+ Tran Thi Lan và Bui Tien Dung (9.2) → đồng hạng 2

##4. Hãy bổ sung thêm một trường graduation_date có kiểu dữ liệu là DATETIME vào bảng student để xác định thời gian tốt nghiệp của từng bạn, trong đó thời gian tốt nghiệp được xác định bởi thời gian hiện tại cộng với số hạng tương ứng của bạn đó tính theo điểm số.

a, Thêm cột graduation_date

In [21]:
conn.execute("""
ALTER TABLE student
ADD COLUMN graduation_date DATETIME
""")

KL: Câu lệnh này thêm một cột mới tên là graduation_date vào bảng student.

b, Xác định thời gian tốt nghiệp của từng bạn, trong đó thời gian tốt nghiệp được xác định bởi thời gian hiện tại cộng với số hạng tương ứng của bạn đó tính theo điểm số.

In [22]:
query_update = """
WITH ranked_list AS (
    SELECT student_id,
           RANK() OVER (ORDER BY score DESC) as rnk
    FROM student
)
UPDATE student
SET graduation_date = datetime('now', '+' || (
    SELECT rnk FROM ranked_list
    WHERE ranked_list.student_id = student.student_id
) || ' months')
"""
#tính mỗi hạng theo tháng
conn.execute(query_update)

KL:
- Tính ngày tốt nghiệp (graduation_date) cho từng sinh viên dựa theo thứ hạng điểm (RANK()).
- Sinh viên điểm cao hơn sẽ được tốt nghiệp sớm hơn (vd: hạng 1 + thêm 1 tháng, hạng 2 + thêm 2 tháng) nghĩa là ngày hiện tại + số tháng = thứ hạng của sinh viên.


In [23]:
import pandas as pd

query = """
SELECT student_id, name, score, graduation_date
FROM student
ORDER BY score DESC
"""

pd.read_sql(query, conn)

,student_id,name,score,graduation_date
0,2,Tran Thi Lan,9.2,2026-05-02 07:09:56
1,7,Bui Tien Dung,9.2,2026-05-02 07:09:56
2,3,Pham Van Nam,7.9,2026-07-02 07:09:56
3,9,Duong Huu Phuc,7.2,2026-08-02 07:09:56
4,10,Cao Thi Hanh,7.0,2026-09-02 07:09:56
5,1,Nguyen Minh Hoang,6.7,2026-10-02 07:09:56


KL: graduation_date phản ánh thứ hạng học lực: điểm cao → tốt nghiệp sớm, điểm thấp → tốt nghiệp muộn.
- Tran Thi lan và Bui Tien Dung đồng hạng (hạng 1: 9.2) → tốt nghiệp sớm nhất: 2026-05-01.
- Pham Van Nam (hạng 3: 7.9) +3 tháng → dự đoán tốt nghiệp: 2026-07-01.
- Duong Huu Phuc (hạng 4: 7.2) +4 tháng → dự đoán tốt nghiệp: 2026-08-01.
- Cao Thi Hanh (hạng 5: 7.0) +5 tháng → dự đoán tốt nghiệp: 2026-09-01.
- Nguyen Minh Hoang (hạng 6: 6.7) +6 tháng → dự đoán tốt nghiệp: 2026-10-01.